# Jour 2 · Neurone, loss et apprentissage

## Objectifs

- comprendre poids, biais et activation sans calcul avancé
- expliquer loss, optimiseur, batch et époque
- observer un neurone apprendre une relation simple


## Un neurone artificiel est un petit calcul

Il reçoit plusieurs nombres, multiplie chaque entrée par un poids, ajoute un biais puis applique éventuellement une fonction d'activation. Les poids représentent la contribution apprise de chaque entrée ; le biais permet de décaler la réponse.

Nous n'avons pas besoin de démontrer les dérivées pour comprendre la mécanique générale.


![Trois entrées pondérées allant vers un neurone](../ressources/illustrations/jour_02/01_neurone.png)

*Les poids et le biais sont ajustés par l’entraînement ; ils ne sont pas choisis manuellement ligne par ligne.*


![Calcul pas à pas dans un neurone artificiel](../ressources/illustrations/jour_02/01_neurone_calcul.png)

Chaque entrée `x` est multipliée par un poids `w`. Un poids positif pousse la somme dans le même sens que l’entrée ; un poids négatif agit dans le sens opposé. Le **biais** ajoute une valeur indépendante des entrées et permet de déplacer le point à partir duquel le neurone réagit.

La somme obtenue avant l’activation est parfois appelée **pré-activation**. La fonction d’activation transforme ensuite cette valeur. Dans le schéma, ReLU reçoit `0,7` et renvoie `0,7`.

Un neurone isolé reste un calcul simple. La puissance d’un réseau vient de nombreux neurones organisés en couches, dont les sorties deviennent les entrées des couches suivantes.


## Pourquoi une activation ?

**ReLU** renvoie zéro pour une entrée négative et conserve une entrée positive. Par exemple : `−2 → 0`, `0 → 0`, `2 → 2`. Le graphique montre la cassure en zéro. Cette transformation est **non linéaire** : elle ne suit pas une seule droite sur tout l'intervalle.

Empiler uniquement des couches qui additionnent et multiplient revient encore à une relation linéaire. Les activations permettent aux couches de représenter des relations plus variées.

![Courbe ReLU et trois exemples d entrée sortie](../ressources/illustrations/jour_02/01_relu_courbe.svg)

Petite vérification orale : que renvoie ReLU pour `−0,5` ? Et pour `3` ? Retrouvez ces deux positions sur les axes.

## La boucle d'apprentissage

Le réseau produit une sortie ; la **loss**, ou perte, mesure son écart à la réponse attendue. La rétropropagation calcule les gradients, puis l'**optimiseur** modifie légèrement les poids et le biais. Nous répétons cette boucle sur de nombreux exemples.


![Boucle batch prédiction loss optimiseur et poids](../ressources/illustrations/jour_02/01_boucle_apprentissage.png)

*L’optimiseur ne connaît pas le métier ; il cherche seulement à réduire la loss définie.*


## Comment l'erreur revient vers les poids

![Étapes de la rétropropagation](../ressources/illustrations/jour_02/01_retropropagation_parametres.svg)

La **rétropropagation** calcule comment la loss varie lorsque chaque paramètre change légèrement. Elle produit les gradients ; elle ne modifie pas elle-même les poids.

1. La **propagation avant** calcule une prédiction à partir des entrées.
2. La **loss** mesure l'écart entre cette prédiction et la bonne réponse.
3. La rétropropagation calcule, couche après couche et en sens inverse, dans quelle direction chaque poids devrait évoluer.
4. L'**optimiseur** applique de petites corrections aux poids.

Le **gradient** indique la sensibilité de la loss à un paramètre : son signe donne une direction et sa taille indique l'influence locale d'une petite modification. Il n'est pas nécessaire de calculer les dérivées à la main pour comprendre l'idée : le réseau teste sa prédiction, mesure son erreur, calcule les gradients en arrière, puis l'optimiseur corrige les paramètres. Les entrées et les réponses attendues restent fixes.


## Descendre progressivement la loss

![Effet du learning rate sur la descente de la loss](../ressources/illustrations/jour_02/01_learning_rate.png)

L’optimisation ressemble à une descente dans un paysage dont l’altitude représente la loss. Le **gradient** indique la pente locale. L’optimiseur choisit une direction qui devrait réduire la loss, puis avance d’un certain pas.

Le **learning rate**, ou taux d’apprentissage, règle la taille de ce pas :

- trop petit, l’apprentissage peut être très lent ;
- adapté, il progresse régulièrement ;
- trop grand, il peut dépasser une bonne zone, osciller ou diverger.

Adam adapte une partie de cette dynamique automatiquement, mais son learning rate reste un hyperparamètre important. Une loss qui devient `NaN` ou explose peut signaler des données mal préparées ou un pas beaucoup trop grand.


## Batch, itération et époque

Il serait possible de calculer une correction après chaque exemple, mais les mises à jour seraient très instables. À l’inverse, utiliser toutes les données avant chaque correction peut demander beaucoup de mémoire.

Un **batch** est le compromis : le réseau traite un petit lot, moyenne son erreur puis met à jour ses poids. Une **itération** correspond à une mise à jour sur un batch. Une **époque** est terminée lorsque tous les exemples du train ont été parcourus une fois.

Avec 120 exemples et `batch_size=16`, une époque contient environ huit itérations. Au début de l’époque suivante, les exemples peuvent être mélangés et revus avec les nouveaux poids.


![Huit exemples répartis en batchs](../ressources/illustrations/jour_02/01_batch_epoque.png)

*Un batch précède une correction ; une époque signifie que tous les exemples du train ont été vus une fois.*


### Vérifier batch et époque avant de continuer

Avec 10 exemples et des lots de 4, on traite des lots de 4, 4 et 2 exemples : cela fait trois corrections par époque. Le dernier lot peut être plus petit. Avec deux époques, chaque exemple est revu deux fois, mais les poids ont changé entre les passages.


## Une expérience volontairement minuscule

Nous demandons à un seul neurone d'apprendre la relation `y = 2x + 1`. L'objectif n'est pas la performance, mais de voir que les poids partent d'une valeur initiale puis se rapprochent d'une relation utile.

### Les objets visibles dans le code

**TensorFlow** réalise les calculs numériques et l'optimisation. **Keras** est son interface de haut niveau. Un modèle `Sequential` empile les couches dans l'ordre écrit. `Dense(1)` crée ici un neurone de sortie.

La loss `mse`, ou erreur quadratique moyenne, met les écarts au carré puis calcule leur moyenne. **Adam** est l'optimiseur : il adapte progressivement la taille de ses corrections. Le `learning_rate` contrôle l'ampleur des pas ; trop grand peut rendre l'apprentissage instable, trop petit peut le rendre très lent.


## Préparer les outils

Un **import** rend un outil Python disponible. `as np`, `as plt` et `as tf` lui donnent un nom court. NumPy gère les tableaux ; Matplotlib les affiche ; TensorFlow entraîne les réseaux. Les premières lignes configurent les messages et le calcul sur CPU. Elles se recopient telles quelles et ne font pas partie du modèle.

La **graine aléatoire** fixe le point de départ du hasard. `42` est un choix arbitraire : il facilite les comparaisons sans garantir des résultats exactement identiques sur toutes les machines.


In [ ]:
import os  # Réglages de l'environnement avant le chargement de TensorFlow.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")  # Réduit les messages techniques.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")  # Utilise le CPU pour ce petit cours.

import numpy as np  # Tableaux et calculs numériques.
import matplotlib.pyplot as plt  # Graphiques et images.
import tensorflow as tf  # Réseaux et calcul automatique des gradients.

tf.keras.utils.set_random_seed(42)  # Fixe le hasard pour comparer les expériences.
try:
    # Limite le nombre de threads utilisés par TensorFlow.
    # Un thread par opération suffit ; ce réglage limite la consommation du cours.
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass  # Ce réglage ne peut plus changer si TensorFlow a déjà commencé à calculer.
print("TensorFlow :", tf.__version__)


## 1. Construire une relation connue

`np.linspace(-1, 1, 120)` construit 120 nombres régulièrement espacés entre −1 et 1. `x` est l'entrée ; `y` la réponse calculée par la relation connue `2x + 1`. `float32` est un format de nombres décimaux adapté au calcul du réseau.

Nous connaissons la règle uniquement pour vérifier ce que le neurone apprend. Il reçoit des couples de nombres, pas la formule Python comme instruction.


In [ ]:
x = np.linspace(-1, 1, 120, dtype="float32")  # Exemples répartis entre −1 et 1.
y = (2 * x + 1).astype("float32")  # Réponse connue pour chacun des exemples.
plt.scatter(x, y, s=12)  # Un point représente un couple entrée/réponse.
plt.xlabel("entrée x")
plt.ylabel("réponse y")
plt.title("La relation que le neurone doit retrouver")
plt.show()


## 2. Créer le neurone et choisir sa mesure d'erreur

`Input(shape=(1,))` attend un nombre par exemple. `Dense(1)` crée un neurone avec un poids et un biais : il calcule `poids × x + biais`. Ici, aucune activation n'est utilisée, car la réponse attendue est une relation linéaire.

**MSE** signifie moyenne des erreurs au carré : pour des écarts de 1 et −2, elle vaut `(1² + (−2)²) / 2 = 2,5`. Le carré rend les écarts positifs et pénalise davantage les grands écarts. `compile` indique comment apprendre ; cette commande n'entraîne pas encore le neurone.


In [ ]:
neuron = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),  # Un nombre d'entrée par exemple.
    tf.keras.layers.Dense(1),  # Un poids et un biais à apprendre, sans activation.
])
neuron.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),  # Pas d'apprentissage pour Adam.
    loss="mse",  # Compare prédiction et réponse avec l'erreur quadratique moyenne.
)
print("Nombre de paramètres :", neuron.count_params())  # Deux : un poids et un biais.


## 3. Lancer l'entraînement

`fit(x, y)` fournit les couples entrée/réponse. Le neurone parcourt les 120 exemples pendant 80 époques au maximum, par lots de 16. L'objet `history` garde l'évolution de la loss après chaque époque. Cet exemple minuscule illustre le mécanisme ; le prochain réseau séparera train, validation et test.


In [ ]:
history = neuron.fit(
    x, y,  # Entrées et réponses associées, dans le même ordre.
    epochs=80,  # Nombre de passages sur les exemples.
    batch_size=16,  # Nombre d'exemples traités avant une correction.
    verbose=0,  # Masque le journal détaillé, sans modifier les calculs.
)
print("Époques réalisées :", len(history.history["loss"]))


## Lire le résultat et la courbe

Les poids sont initialisés avec de petites valeurs avant la première époque. Ils ne connaissent donc pas encore la relation `y = 2x + 1`.

La courbe affiche la MSE après chaque époque. Une baisse rapide au début signifie que le neurone corrige ses plus grosses erreurs. Une stabilisation indique que les améliorations deviennent petites. Une courbe parfaitement décroissante n’est pas obligatoire : avec des batchs, de légères variations peuvent apparaître.

Le poids devrait se rapprocher de 2 et le biais de 1. Ils ne seront pas nécessairement exacts : l’apprentissage procède par corrections successives et s’arrête après un nombre fini d’époques.

Une loss qui diminue sur le train prouve seulement que le modèle reproduit mieux les exemples vus. Elle ne suffit pas à démontrer qu’il généralisera à de nouvelles données. C’est pour cela que les réseaux suivants utiliseront une validation et un test.


In [ ]:
plt.plot(history.history["loss"])  # Une valeur d'erreur par époque.
plt.xlabel("époque")
plt.ylabel("MSE")
plt.title("L'erreur sur les exemples vus pendant l'apprentissage")
plt.show()

weight, bias = neuron.layers[0].get_weights()  # Récupère le poids et le biais appris.
print("poids appris :", round(float(weight[0, 0]), 3))  # Case du seul poids.
print("biais appris :", round(float(bias[0]), 3))  # Première et seule valeur du biais.


## 4. Tester dans la zone observée, puis en dehors

Le neurone a vu des valeurs entre −1 et 1. Tester `x = 0,5` est une **interpolation** : la nouvelle entrée est dans cette zone. Tester `x = 4` est une **extrapolation** : l'entrée est en dehors de la zone d'apprentissage.

Pour prédire un exemple, nous utilisons `np.array([[0.5]], dtype="float32")`. Les doubles crochets donnent un tableau de forme `(1, 1)` : un exemple, contenant un nombre. `predict` renvoie également un tableau ; `[0, 0]` prend la première sortie du premier exemple.


## À vous de jouer — prédire une valeur dans la zone connue

Complétez la création du tableau pour `x = 0,5`. Prévoyez d'abord la réponse avec `2 × 0,5 + 1`, puis comparez-la à la prédiction. Le code de lecture est fourni.


In [ ]:
nouvelle_entree = None  # À compléter : np.array([[0.5]], dtype="float32").
if nouvelle_entree is None:
    print("Créez le tableau de forme (1, 1).")
else:
    prediction = float(neuron.predict(nouvelle_entree, verbose=0)[0, 0])
    print("prédiction :", round(prediction, 3), "réponse connue :", 2 * 0.5 + 1)


## À vous de jouer — extension facultative — extrapoler à x = 4

Une fois l'interpolation comprise, faites le même test avec `x = 4`. La réponse de la relation connue vaut `9`. Pourquoi une réussite sur cet exemple ne prouve-t-elle pas que tous les modèles extrapolent correctement ?


In [ ]:
entree_lointaine = None  # Facultatif : créez le tableau pour x = 4.
if entree_lointaine is None:
    print("Extension facultative : comparez une entrée hors de [−1, 1].")
else:
    print("prédiction :", float(neuron.predict(entree_lointaine, verbose=0)[0, 0]))
    print("réponse connue :", 9.0)
